<a href="https://colab.research.google.com/github/sechali/buku_ajar_ai/blob/main/notebooks/part-13/AI_Modul_13.4.2_Praktikum_Monitoring_Berbasis_CCTV.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Modul 13.4.2: Praktikum Sistem Monitoring CCTV Cerdas (Intrusion Detection)
**Tujuan Pembelajaran:**
1. Mengimplementasikan algoritma Ray Casting (Point-in-Polygon) murni untuk deteksi zona perimeter sembarang.
2. Membangun sistem pelacakan waktu tinggal (*Dwell Time Tracking*) untuk memvalidasi peristiwa intrusi.
3. Mensimulasikan pemantauan CCTV keamanan pabrik kelapa sawit dengan visualisasi zona transparan.

In [ ]:
import cv2
import numpy as np
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches

print("Modul praktikum monitoring CCTV cerdas siap!")

### Bagian 1: Implementasi Algoritma Geometris Ray Casting

In [ ]:
def point_in_polygon(point, poly_pts):
    x, y = point
    n = len(poly_pts)
    inside = False
    p1x, p1y = poly_pts[0]
    for i in range(n + 1):
        p2x, p2y = poly_pts[i % n]
        if y > min(p1y, p2y):
            if y <= max(p1y, p2y):
                if x <= max(p1x, p2x):
                    if p1y != p2y:
                        xinters = (y - p1y) * (p2x - p1x) / (p2y - p1y) + p1x
                    if p1x == p2x or x <= xinters:
                        inside = not inside
        p1x, p1y = p2x, p2y
    return inside

# Definisi zona bahaya poligon (segi empat miring di pabrik)
hazard_zone = [(100, 100), (300, 80), (340, 260), (120, 280)]

# Uji titik dalam dan luar zona
pt_inside = (200, 150)
pt_outside = (50, 50)
res_in = point_in_polygon(pt_inside, hazard_zone)
res_out = point_in_polygon(pt_outside, hazard_zone)

print(f"Titik {pt_inside} di dalam zona? {res_in}")
print(f"Titik {pt_outside} di dalam zona? {res_out}")
assert res_in is True and res_out is False, "Logika ray casting keliru!"
print("[VALIDASI SUKSES] Algoritma Ray Casting teruji benar secara analitis!")

### Bagian 2: Simulasi Pelacakan CCTV dan Penilaian Waktu Tinggal (Dwell Time)

In [ ]:
class CCTVZoneSimulator:
    def __init__(self, zone, dwell_threshold_frames=4):
        self.zone = zone
        self.dwell_threshold = dwell_threshold_frames
        self.track_frames_in_zone = {}
        self.alarm_triggered = set()
        
    def update(self, current_tracks):
        alarms = []
        for trk_id, (x, y) in current_tracks.items():
            inside = point_in_polygon((x, y), self.zone)
            if inside:
                self.track_frames_in_zone[trk_id] = self.track_frames_in_zone.get(trk_id, 0) + 1
                if self.track_frames_in_zone[trk_id] >= self.dwell_threshold and trk_id not in self.alarm_triggered:
                    self.alarm_triggered.add(trk_id)
                    alarms.append(f"ALARM: Objek #{trk_id} bertahan di zona bahaya selama {self.track_frames_in_zone[trk_id]} frame!")
            else:
                # Reset jika keluar
                if trk_id in self.track_frames_in_zone:
                    del self.track_frames_in_zone[trk_id]
        return alarms

sim = CCTVZoneSimulator(hazard_zone, dwell_threshold_frames=3)

# Simulasi 6 frame pergerakan 2 orang:
# Orang 1: Masuk zona dan bertahan (Penyusup)
# Orang 2: Berjalan di luar zona (Pekerja aman)
all_alarms = []
for f in range(6):
    coords = {
        101: (200, 150),                  # Selalu di dalam zona
        102: (50 + f * 10, 40 + f * 5)   # Bergerak di luar zona
    }
    ev = sim.update(coords)
    if len(ev) > 0:
        all_alarms.extend(ev)

print(f"\nTotal Alarm Terpicu: {len(all_alarms)}")
for a in all_alarms:
    print(" ", a)
assert len(all_alarms) == 1, "Alarm harusnya terpicu tepat 1 kali untuk penyusup #101!"
print("[VALIDASI SUKSES] Penilaian Dwell Time berhasil membedakan penyusup dari pekerja biasa!")

### Bagian 3: Visualisasi Tampilan CCTV Cerdas dengan Poligon ROI Transparan

In [ ]:
cctv_canvas = np.full((360, 480, 3), 40, dtype=np.uint8)
# Gambar lantai stasiun pabrik
cv2.line(cctv_canvas, (0, 300), (480, 300), (80, 80, 80), 2)

# Overlay Zona Poligon Berwarna Merah (Karena ada alarm)
zone_pts = np.array(hazard_zone, dtype=np.int32)
overlay = cctv_canvas.copy()
cv2.fillPoly(overlay, [zone_pts], (0, 0, 220)) # Merah transparan BGR
cv2.addWeighted(overlay, 0.35, cctv_canvas, 0.65, 0, cctv_canvas)
cv2.polylines(cctv_canvas, [zone_pts], True, (0, 0, 255), 2)

# Gambar Posisi Penyusup #101
cv2.circle(cctv_canvas, (200, 150), 12, (0, 0, 255), -1)
cv2.rectangle(cctv_canvas, (180, 110), (220, 190), (0, 0, 255), 2)
cv2.putText(cctv_canvas, "INTRUDER #101", (160, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 2)

# Gambar Posisi Pekerja Aman #102
cv2.circle(cctv_canvas, (100, 65), 10, (0, 255, 0), -1)
cv2.rectangle(cctv_canvas, (85, 30), (115, 100), (0, 255, 0), 2)
cv2.putText(cctv_canvas, "STAFF #102", (70, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)

# Banner Status Sistem
cv2.putText(cctv_canvas, "CCTV-04: STASIUN STERILIZER [SECURITY BREACH]", (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255, 255, 255), 2)

plt.figure(figsize=(7, 5))
plt.imshow(cv2.cvtColor(cctv_canvas, cv2.COLOR_BGR2RGB))
plt.title('Tampilan Monitor CCTV AI: Deteksi Intrusi Poligon & Pelacakan Objek')
plt.axis('off')
plt.tight_layout()
plt.savefig('praktikum_cctv_smart_monitoring_13_4_2.png', dpi=200)
plt.close()
print("[OK] Tangkapan layar CCTV pintar disimpan sebagai 'praktikum_cctv_smart_monitoring_13_4_2.png'")